# CAPTCHA OCR

Educational OCR project for CAPTCHA recognition.

## 1. Data preparation

This section collects image paths, extracts labels from filenames, builds the character vocabulary, defines a PyTorch dataset that loads images with Pillow, and creates a reproducible 80/20 train/test split.

In [1]:
from pathlib import Path
import random

import torch
from torch.utils.data import Dataset, Subset
from PIL import Image

SEED = 42
DATA_DIR = Path("data/samples")
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}

random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
# проверка cuda
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA GeForce RTX 5090 Laptop GPU


In [3]:
print(torch.version.cuda)

12.8


In [ ]:
image_paths = sorted(
    path for path in DATA_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

labels = [path.stem for path in image_paths]
captcha_length = len(labels[0]) if labels else 0

assert image_paths, f"No images found in {DATA_DIR}"
assert all(len(label) == captcha_length for label in labels), "All labels must have the same length"

alphabet = sorted(set("".join(labels)))
char_to_idx = {char: idx for idx, char in enumerate(alphabet)}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

print(f"Images: {len(image_paths)}")
print(f"CAPTCHA length: {captcha_length}")
print(f"Alphabet size: {len(alphabet)}")
print(f"Alphabet: {''.join(alphabet)}")

In [ ]:
class CaptchaDataset(Dataset):
    def __init__(self, image_paths, char_to_idx, transform=None):
        self.image_paths = list(image_paths)
        self.char_to_idx = char_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = image_path.stem

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            if self.transform is not None:
                image = self.transform(image)

        target = torch.tensor([self.char_to_idx[char] for char in label], dtype=torch.long)
        return image, target, label, image_path

In [ ]:
dataset = CaptchaDataset(image_paths=image_paths, char_to_idx=char_to_idx)

indices = list(range(len(dataset)))
rng = random.Random(SEED)
rng.shuffle(indices)

train_size = int(0.8 * len(indices))
train_indices = indices[:train_size]
test_indices = indices[train_size:]

train_dataset = Subset(dataset, train_indices)
test_dataset = Subset(dataset, test_indices)

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

In [ ]:
sample_image, sample_target, sample_label, sample_path = dataset[0]

print(f"Sample path: {sample_path}")
print(f"Sample label: {sample_label}")
print(f"Encoded target: {sample_target.tolist()}")
print(f"Image mode: {sample_image.mode}")
print(f"Image size: {sample_image.size}")

## 2. Model creation and training

To be implemented.

## 3. CER evaluation

To be implemented.

## 4. Error analysis

To be implemented.

## 5. Conclusions

To be implemented.